# BTK Datathon 2026 — Career Success Score Prediction

**Task.** Predict `career_success_score` (continuous, 0–100) for 10,000 students from 40 numeric
features, 5 categoricals, and a Turkish free-text mentor-feedback field. **Metric: MSE.**

This notebook is the **final writeup**: it loads the artifacts produced by the reproducible
pipeline in `src/` + `scripts/` and walks through the data, the modelling, the blend, and the
signal-floor analysis that decided the final submission. To regenerate everything from scratch:

```bash
./run.sh setup-gpu          # deps (CPU: ./run.sh setup)
./run.sh all                # every config in configs/  (exp001 .. exp021)
./run.sh blend --method ridge --tag blend_full_ridge
./run.sh submit blend_full_ridge
```

Per-experiment narrative lives in `EXPERIMENTS.md`; the day-by-day log in `docs/progress/`.

**Best result: ridge-stack blend, CV MSE 73.61 → public LB 82.96.**

## 1. Setup

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')   # run from notebooks/ ; or '.' from repo root
import numpy as np, pandas as pd
from pathlib import Path

from src.data import TARGET, TEXT_COL, ID_COL, load_raw, fold_array
from src.cv import mse, rmse
from src.utils import ARTIFACTS, RESULTS_CSV

ROOT = Path('..') if Path('../src').exists() else Path('.')
pd.set_option('display.max_colwidth', 90)

## 2. Data

In [ ]:
train, test, _ = load_raw()
y = train[TARGET].to_numpy(float)
print(f'train: {train.shape} | test: {test.shape}')
print(f'target: mean={y.mean():.2f} std={y.std():.2f} | %at100={np.mean(y==100)*100:.1f}% | %at0={np.mean(y==0)*100:.2f}%')
train[[TARGET, TEXT_COL]].head(3)

## 3. Key EDA findings

Full EDA code: `reports/eda/deep_eda.py`. The findings that drove every modelling decision:

1. **The text verbalizes the profile.** The Turkish `mentor_feedback_text` is LLM-generated; words
   like *mükemmel* (excellent) → mean target 91.8, *sınırlı* (limited) → 67.0. OOF TF-IDF
   ridge/logistic meta-features are the single biggest win (−5.5 CV MSE).
2. **Ceiling effect.** 7.7% of train has y=100 exactly; a P(y==100) classifier reaches AUC ~0.93.
3. **Covariate shift is temporal only.** Adversarial AUC 0.65 with year features, ~0.49 without.
   The test skews to 2024–2026 (62%), where target means are lower.
4. **Missingness is signal.** NA rows score lower; per-column NA flags + row NA count are features.
5. **Top SHAP interaction:** `application_year × project_quality_score`.

In [ ]:
from IPython.display import Image, display
figs = ['target_distribution','text_ridge_coefficients','shap_interaction_heatmaps','baseline_residuals']
for f in figs:
    p = ROOT/'reports'/'figures'/f'{f}.png'
    if p.exists(): display(Image(filename=str(p)))

## 4. Pipeline & CV protocol

Every experiment is one `configs/<exp_id>.yaml` run through `scripts/run_experiment.py`, which
obeys a non-negotiable protocol so the OOF predictions are valid for stacking:

- **One fixed fold file** (`data/processed/folds.csv`), StratifiedKFold(5) on target deciles, seed 42 — never re-split.
- **Every fitted transform** (TF-IDF, target encoding, kNN-target, scalers, fine-tune) fits **inside the training fold only**.
- **All predictions clipped to [0,100]**; MSE/RMSE plus year-sliced and non-ceiling RMSE logged.
- **Artifacts persisted** per experiment (`oof_{exp}.npy`, `test_{exp}.npy`) for the blend.

Feature groups (`src/features/`): tabular FE (NA flags, role–skill alignment, ratios, time,
OOF target encoding), classic text (lexicon/stats), TF-IDF ridge/logistic meta, e5-large
embeddings (SVD + kNN-target), and transformer fine-tune OOFs (BERT / XLM-R).

## 5. Results — every experiment (from `results.csv`)

In [ ]:
res = pd.read_csv(RESULTS_CSV)
latest = res.sort_values('timestamp').groupby('exp_id', as_index=False).last()
cols = ['exp_id','description','cv_mse','cv_rmse','rmse_year_2024plus']
latest = latest[cols].sort_values('cv_mse').reset_index(drop=True)
latest['description'] = latest['description'].str.slice(0,60)
latest

## 6. The final blend (ridge stacker)

The blend is a fold-safe, non-negative **Ridge stacker** over the OOF matrix (it beats convex-weight
blending every time). A weak member simply gets ~0 weight, so adding members never hurts.
Reproduced here from saved OOFs:

In [ ]:
from sklearn.linear_model import Ridge
from src.data import N_FOLDS
folds = fold_array(train)
members = [p.stem.removeprefix('oof_') for p in sorted((ROOT/'artifacts').glob('oof_exp*.npy'))]
members = [m for m in members if m != 'exp017']   # pseudo-label OOF is optimistic; excluded
oofs  = np.column_stack([np.load(ROOT/'artifacts'/f'oof_{m}.npy')  for m in members])

stack = np.zeros(len(y))
for f in range(N_FOLDS):
    tr, va = folds != f, folds == f
    r = Ridge(alpha=1.0, positive=True).fit(oofs[tr], y[tr])
    stack[va] = r.predict(oofs[va]).clip(0,100)
print(f'blend members: {len(members)}')
print(f'blend CV  MSE = {mse(y, stack):.3f} | RMSE = {rmse(y, stack):.4f}')
print(f'best single   = {min(mse(y, oofs[:,i]) for i in range(len(members))):.3f} MSE')

## 7. Why we stopped — the signal floor

Full reproducible analysis: `reports/eda/floor_analysis.py` (7 diagnostics, all from saved
artifacts, leak-free). The decisive one: **within-year R² is flat at ~0.69**, so late years are
harder only because their target *variance* is larger — not because we model them worse. The
CV→LB gap is 100% the test's late-year skew, and `target = g(features) + irreducible noise`.

In [ ]:
oof = np.load(ROOT/'artifacts'/'oof_blend_full_ridge.npy') if (ROOT/'artifacts'/'oof_blend_full_ridge.npy').exists() else stack
res_ = y - oof
yr = train['application_year'].to_numpy()
rows = []
for Y in sorted(np.unique(yr)):
    m = yr == Y
    rows.append({'year':Y, 'n':int(m.sum()), 'tgt_var':round(float(np.var(y[m])),1),
                 'oof_mse':round(float(np.mean(res_[m]**2)),1),
                 'R2':round(1-np.mean(res_[m]**2)/np.var(y[m]),3)})
pd.DataFrame(rows)

In [ ]:
for f in ['r2_by_year','year_error_decomposition','blend_member_corr','public_lb_noise']:
    p = ROOT/'reports'/'figures'/f'{f}.png'
    if p.exists(): display(Image(filename=str(p)))

## 8. Final submission

`blend_full_ridge` was best on both CV and LB and is the locked final entry. Sanity vs the train target:

In [ ]:
sub = pd.read_csv(ROOT/'submissions'/'sub_blend_full_ridge.csv')
p = sub[TARGET].to_numpy()
print(f'rows={len(sub)} | mean={p.mean():.2f} (train {y.mean():.2f}) | std={p.std():.2f} (train {y.std():.2f})')
print(f'min={p.min():.1f} max={p.max():.1f} | %at100={np.mean(p==100)*100:.2f}% (train {np.mean(y==100)*100:.2f}%)')
sub.head()

## 9. Conclusion

- A reproducible, leakage-safe pipeline: fixed folds, fold-internal transforms, per-experiment
  artifacts, an append-only results log, and a single `./run.sh` entry point.
- The text is the biggest lever (TF-IDF ridge/logistic meta + transformer fine-tune OOFs); the
  blend is a fold-safe Ridge stacker.
- **CV is a faithful LB proxy** here (constant +9.3 offset from the test's late-year skew; ranking
  preserved across 5 final blend variants), so all model selection trusts `cv_mse`.
- Seven independent diagnostics show the modelling frontier is closed for the levers explored:
  `target = g(features) + irreducible noise`, within-year R² ≈ 0.69, blend members 0.95-correlated,
  and a fresh aspect-based text extraction adds nothing the existing representations didn't capture.
- **Final: `blend_full_ridge`, CV 73.61 / LB 82.96** — a top-tier, well-calibrated, non-overfit solution.